# S&P 500 Derivatives Pricing — Project Cheat Sheet

**Single-notebook reference for every pricing engine in `src/engines/`.**
Each section gives you (1) the math, (2) plain-English *when to use this*,
(3) a hand-rolled implementation, and (4) the equivalent QuantLib call —
all driven by **live S&P 500 data from yfinance**.

## Table of contents

| § | Topic | Engine file | QuantLib class |
|---|-------|-------------|----------------|
| 0 | QuantLib quick-connect cheat sheet | — | (reference table) |
| 1 | Live S&P 500 trade dict (yfinance) | — | — |
| 2 | Black–Scholes (European, closed-form) | `black_scholes.py` | `AnalyticEuropeanEngine` |
| 3 | Binomial tree (American, CRR / LR) | `quantlib_engine.price_american_ql` | `BinomialVanillaEngine` |
| 4 | Monte Carlo + Longstaff–Schwartz | `monte_carlo_lsm.py` | `MCAmericanEngine` |
| 5 | Knock-out / knock-in barriers | `knockout.py`, `quantlib_engine.price_knockout_ql` | `AnalyticBarrierEngine`, `FdBlackScholesBarrierEngine` |
| 6 | Discrete-cash dividends (American) | `quantlib_engine.price_american_discrete_div_ql` | `FdBlackScholesVanillaEngine` |
| 7 | Greeks: analytic vs bump vs FDM | `quantlib_engine.greeks_*` | (multiple) |
| 8 | Engine router decision tree | `router.py` | — |
| 9 | One-shot summary table | — | — |

> **How to read the notebook:** start at §0, run §1 to populate the live
> `TRADE` dict, then any later section is self-contained — they all read
> from `TRADE`. Run top-to-bottom for a full price comparison.


---
# §0 — QuantLib Quick-Connect Cheat Sheet

Every QL pricing call in this project follows the **same 4-step pattern**.
Memorise this and you can build any engine in 10 lines.

### The pattern

```python
import QuantLib as ql

# 1) Set the global evaluation date (today by default)
today = ql.Date.todaysDate()
ql.Settings.instance().evaluationDate = today

# 2) Build the GBM process (spot + r-curve + q-curve + vol surface)
spot = ql.QuoteHandle(ql.SimpleQuote(S))
r_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, r, ql.Actual365Fixed()))
q_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, q, ql.Actual365Fixed()))
v_ts = ql.BlackVolTermStructureHandle(
    ql.BlackConstantVol(today, ql.UnitedStates(ql.UnitedStates.NYSE),
                        sigma, ql.Actual365Fixed()))
process = ql.GeneralizedBlackScholesProcess(spot, q_ts, r_ts, v_ts)

# 3) Build the instrument (payoff + exercise)
payoff   = ql.PlainVanillaPayoff(ql.Option.Put, K)
exercise = ql.EuropeanExercise(today + int(T*365))   # or AmericanExercise(today, maturity)
option   = ql.VanillaOption(payoff, exercise)

# 4) Pick the engine and price
option.setPricingEngine(ql.AnalyticEuropeanEngine(process))
price = option.NPV()
```

### Engine reference — which class for which payoff?

| Payoff | QuantLib engine | Repo function (`src/engines/`) | Use when |
|---|---|---|---|
| **European vanilla** | `AnalyticEuropeanEngine(process)` | `quantlib_engine.greeks_ql(..., is_american=False)` | Closed-form, instant. Default for any European call/put. |
| **American vanilla** | `BinomialVanillaEngine(process, "lr", N)` | `quantlib_engine.price_american_ql` | American exercise w/ continuous dividend yield. LR is O(1/N²) — converges smoothly. |
| **American + cash divs** | `FdBlackScholesVanillaEngine(process, divSched, t, x)` | `quantlib_engine.price_american_discrete_div_ql` | American call/put with discrete cash dividends (single-name equity). Continuous-q can't reproduce the ex-div spot drop. |
| **American (MC)** | `MCAmericanEngine(process, "PseudoRandom", ...)` | `monte_carlo_lsm.price_american` | Sanity-check the binomial tree, or when the payoff is path-dependent on top of early exercise. |
| **Continuous-monitored barrier** | `AnalyticBarrierEngine(process)` | `quantlib_engine.price_knockout_ql` (default) | KO/KI with continuous monitoring under flat vol. Reiner–Rubinstein closed-form. |
| **Barrier + smile** | `FdBlackScholesBarrierEngine(process, t, x, 0, Douglas, True)` | `price_knockout_ql(..., use_local_vol_pde=True)` | Steep skew (e.g. SPX KO call): analytic engine collapses surface to one σ → directionally wrong. FD with Dupire local vol fixes it. |
| **Discretely-monitored barrier** | `AnalyticBarrierEngine` + BGK shift on B | `knockout.bgk_adjusted_barrier` then `price_knockout_ql` | Daily/weekly/monthly fix. Without BGK you over-state KO probability → under-price KO product by 30–80 bp. |
| **American FDM Greeks** | `FdBlackScholesVanillaEngine(process, t, x)` | `quantlib_engine.greeks_american_fdm_ql` | Risk reporting: LR-tree γ has "ghost gamma" — adjacent strikes can differ by 30%. FDM produces smooth Greek surfaces. |

### Conventions used by every engine in this repo
- **Day count:** Actual/365 Fixed
- **Calendar:** `ql.UnitedStates(ql.UnitedStates.NYSE)` for US equity (NOT `ql.TARGET()` — that's EUR)
- **T → days:** `max(int(T*365 + 0.5), 1)` (round-half-up, floor at 1 day)
- **σ:** annualised; vega returned per **1 %** absolute σ move
- **r, q:** continuously-compounded Act/365 zero rates
- **θ:** per **calendar day**, sign convention `∂V/∂t` (positive = value rises with time forward)


---
# §1 — Live S&P 500 trade dict (yfinance)

Pulls **real market data** — spot from `^GSPC`, **tenor-matched** risk-free
from the right Treasury yield (`^IRX` 13W for ≤6 mo, `^FVX` 5Y for 6 mo–3 y,
`^TNX` 10Y for 3 y–15 y, `^TYX` 30Y otherwise), realised vol from one year
of daily closes, and the SPY trailing dividend yield. Falls back to sensible
defaults if yfinance is rate-limited (common on Colab free tier).

> **Why tenor-match the rate?** Discounting a 90-day option with the 10Y
> yield is a 50–100 bp error in `r` when the curve is steep (and even more
> when it's inverted, as it is in 2025–26). The institutional fix is a
> SOFR/OIS curve interpolated to the option's exact `T`; for this cheat
> sheet we approximate by picking the closest available constant-maturity
> point.

The output `TRADE` dict is what every later section prices against.


In [ ]:
# === SETUP: install + import ===
# QuantLib is the only non-Colab default we need; yfinance is usually present.
import sys, subprocess
for pkg in ("QuantLib", "yfinance"):
    try:
        __import__(pkg.lower() if pkg != "QuantLib" else "QuantLib")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
import QuantLib as ql
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm
from datetime import datetime, timedelta

np.random.seed(42)
print(f"QuantLib {ql.__version__}  |  numpy {np.__version__}  |  yfinance {yf.__version__}")


In [ ]:
def _treasury_ticker_for_tenor(T_days: int) -> tuple[str, str]:
    """Pick the yfinance Treasury yield ticker closest to the option's tenor.

    Yahoo only exposes the constant-maturity points: ^IRX (13W ≈ 90d),
    ^FVX (5Y), ^TNX (10Y), ^TYX (30Y). For institutional use, replace with
    a SOFR/OIS curve from FRED or a proper yield-curve service so r is
    interpolated to the exact option tenor — using a 10Y rate to discount
    a 90d option is a 50–100 bp error in r when the curve is steep.
    """
    if T_days <=  180:    return "^IRX", "13-week T-bill"
    if T_days <= 3*365:   return "^FVX", "5-year (proxy for 1–3Y curve)"
    if T_days <= 15*365:  return "^TNX", "10-year"
    return                       "^TYX", "30-year"


def fetch_spx_trade(T_days: int = 90, otm_pct: float = 0.0) -> dict:
    """Build a TRADE dict from live S&P 500 data.

    Parameters
    ----------
    T_days : int
        Days to expiration for the option we will price. The risk-free rate
        is fetched from the tenor-matched Treasury yield (see
        ``_treasury_ticker_for_tenor``).
    otm_pct : float
        Strike offset vs spot. 0.0 = ATM, -0.05 = 5% OTM put strike.

    Returns
    -------
    dict with: S, K, r, sigma, T, q, ref_date, source, rate_tenor
    """
    # Sensible fallbacks if yfinance is throttled. Update if running far from
    # this notebook's authoring date — they're a safety net, not the truth.
    fb = {"S": 5415.23, "r": 0.0432, "sigma": 0.1845, "q": 0.015}

    out = {"ref_date": datetime.now().date().isoformat(), "source": "yfinance"}
    try:
        spx = yf.Ticker("^GSPC").history(period="1y", auto_adjust=False)
        if spx.empty:
            raise RuntimeError("empty ^GSPC")
        out["S"] = float(spx["Close"].iloc[-1])
        # 252-day annualised realised vol from log returns
        log_ret = np.log(spx["Close"] / spx["Close"].shift(1)).dropna()
        out["sigma"] = float(log_ret.std() * np.sqrt(252))
    except Exception as e:
        print(f"[yfinance] ^GSPC failed ({e}); using fallback spot/sigma")
        out["S"], out["sigma"], out["source"] = fb["S"], fb["sigma"], "fallback"

    rate_ticker, rate_label = _treasury_ticker_for_tenor(T_days)
    out["rate_tenor"] = f"{rate_ticker} ({rate_label})"
    try:
        tnx = yf.Ticker(rate_ticker).history(period="1mo")
        # ^IRX/^FVX/^TNX/^TYX all quote in % × 100 (e.g. 4.34 = 4.34%).
        out["r"] = float(tnx["Close"].iloc[-1]) / 100.0
    except Exception as e:
        print(f"[yfinance] {rate_ticker} failed ({e}); using fallback r")
        out["r"], out["source"] = fb["r"], "fallback"

    try:
        spy = yf.Ticker("SPY").info
        # info["dividendYield"] is sometimes missing on Colab; compute from
        # trailing distributions / price as a backup.
        dy = spy.get("dividendYield")
        if dy is None or dy == 0:
            div = yf.Ticker("SPY").dividends.last("365D").sum()
            price = yf.Ticker("SPY").history(period="5d")["Close"].iloc[-1]
            dy = float(div / price) if price else fb["q"]
        out["q"] = float(dy if dy < 0.2 else dy / 100.0)  # yfinance flips units
    except Exception as e:
        print(f"[yfinance] SPY div yield failed ({e}); using fallback q")
        out["q"] = fb["q"]

    out["T"] = T_days / 365.0
    out["K"] = round(out["S"] * (1.0 + otm_pct), 2)
    return out


TRADE = fetch_spx_trade(T_days=90, otm_pct=0.0)

print(f"\nLIVE S&P 500 TRADE  ({TRADE['ref_date']}  source={TRADE['source']})")
print("-" * 60)
for k, v in TRADE.items():
    if isinstance(v, float):
        if k in ("r", "sigma", "q"):
            print(f"  {k:10s} = {v:>10.4%}")
        elif k == "T":
            print(f"  {k:10s} = {v:>10.4f} yrs ({v*365:.0f}d)")
        else:
            print(f"  {k:10s} = {v:>10,.2f}")
    else:
        print(f"  {k:10s} = {v}")


---
# §2 — Black–Scholes (European, closed-form)

### When to use
- Exercise type is **European** (only at expiry)
- Underlying is a **liquid index** (SPX, NDX) where the GBM assumption is
  acceptable and there is no smile-driven mispricing you can't ignore
- You need a **fast** answer (every Greek is closed-form too)
- **Don't use** for American puts on dividend-payers, or anything path-dependent

### Math
$$
C = S\,e^{-qT}\,N(d_1) - K\,e^{-rT}\,N(d_2),\qquad
P = K\,e^{-rT}\,N(-d_2) - S\,e^{-qT}\,N(-d_1)
$$
$$
d_1 = \frac{\ln(S/K) + (r - q + \tfrac12\sigma^2)T}{\sigma\sqrt T},\quad
d_2 = d_1 - \sigma\sqrt T
$$

### Identity to remember
**Put–call parity** under continuous div yield:
$C - P = S\,e^{-qT} - K\,e^{-rT}$.
Use it as your first sanity check whenever a European result looks off.


In [ ]:
# --- MANUAL Black-Scholes (this is what src/engines/black_scholes.py implements) ---
def bs_price(S, K, r, sigma, T, q=0.0, kind="put"):
    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if kind == "call":
        return S*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*np.exp(-q*T)*norm.cdf(-d1)

p_manual_call = bs_price(TRADE["S"], TRADE["K"], TRADE["r"], TRADE["sigma"],
                         TRADE["T"], TRADE["q"], "call")
p_manual_put  = bs_price(TRADE["S"], TRADE["K"], TRADE["r"], TRADE["sigma"],
                         TRADE["T"], TRADE["q"], "put")

print(f"Manual BS  call = ${p_manual_call:.4f}")
print(f"Manual BS  put  = ${p_manual_put:.4f}")

# Put-call parity sanity check
parity_lhs = p_manual_call - p_manual_put
parity_rhs = TRADE["S"]*np.exp(-TRADE["q"]*TRADE["T"]) - TRADE["K"]*np.exp(-TRADE["r"]*TRADE["T"])
print(f"\nParity check: C - P = {parity_lhs:.6f}")
print(f"             S e^-qT - K e^-rT = {parity_rhs:.6f}")
print(f"             diff = {parity_lhs - parity_rhs:.2e}   {'OK' if abs(parity_lhs-parity_rhs)<1e-8 else 'FAIL'}")


In [ ]:
# --- QUANTLIB equivalent (matches AnalyticEuropeanEngine in greeks_ql) ---
def ql_european(S, K, r, sigma, T, q=0.0, kind="put"):
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today
    maturity = today + max(int(T*365 + 0.5), 1)

    spot = ql.QuoteHandle(ql.SimpleQuote(S))
    r_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, r, ql.Actual365Fixed()))
    q_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, q, ql.Actual365Fixed()))
    v_ts = ql.BlackVolTermStructureHandle(
        ql.BlackConstantVol(today, ql.UnitedStates(ql.UnitedStates.NYSE),
                            sigma, ql.Actual365Fixed()))
    process = ql.GeneralizedBlackScholesProcess(spot, q_ts, r_ts, v_ts)

    payoff = ql.PlainVanillaPayoff(ql.Option.Call if kind == "call" else ql.Option.Put, K)
    option = ql.VanillaOption(payoff, ql.EuropeanExercise(maturity))
    option.setPricingEngine(ql.AnalyticEuropeanEngine(process))
    return option.NPV(), {
        "delta": option.delta(), "gamma": option.gamma(),
        "vega": option.vega()/100.0, "theta": option.theta()/365.0,
        "rho":  option.rho()/100.0,
    }

p_ql_call, g_ql_call = ql_european(TRADE["S"], TRADE["K"], TRADE["r"],
                                   TRADE["sigma"], TRADE["T"], TRADE["q"], "call")
p_ql_put,  g_ql_put  = ql_european(TRADE["S"], TRADE["K"], TRADE["r"],
                                   TRADE["sigma"], TRADE["T"], TRADE["q"], "put")

print(f"QuantLib   call = ${p_ql_call:.4f}    diff vs manual = {p_ql_call-p_manual_call:+.2e}")
print(f"QuantLib   put  = ${p_ql_put:.4f}    diff vs manual = {p_ql_put-p_manual_put:+.2e}")
print("\nGreeks (QuantLib, put):")
for k, v in g_ql_put.items():
    print(f"  {k:5s} = {v:>10.4f}")

assert abs(p_ql_put - p_manual_put) < 1e-3, "BS manual vs QL mismatch"
print("\nManual = QuantLib  →  AnalyticEuropeanEngine validated.")


---
# §3 — Binomial tree (American, CRR vs Leisen–Reimer)

### When to use
- Exercise type is **American** (deep-ITM puts, or calls on high-yield names)
- Continuous dividend yield `q` is acceptable (otherwise jump to §6)
- You want a deterministic price (no MC noise) with controllable accuracy

### Math (Cox–Ross–Rubinstein)
At each step Δt = T/N:
$u = e^{\sigma\sqrt{\Delta t}},\; d = 1/u,\; p = \frac{e^{(r-q)\Delta t} - d}{u - d}$.
Roll back: $V_i = e^{-r\Delta t}\bigl[p\,V_{i,\text{up}} + (1-p)\,V_{i,\text{down}}\bigr]$,
then **at every node** take $\max(V_i,\,\text{intrinsic})$ for early exercise.

### Why Leisen–Reimer is what the repo uses
CRR converges as O(1/N) and oscillates. **Leisen–Reimer** centres the tree on the
strike with a Peizer–Pratt inversion — convergence is O(1/N²), smooth, and
removes the saw-tooth pricing across maturities. `BinomialVanillaEngine(..., "lr", N)`
requires **N odd** so the strike sits on a node — the repo rounds up internally.


In [ ]:
# --- MANUAL CRR tree (American put) ---
def crr_american(S, K, r, sigma, T, q=0.0, N=500, kind="put"):
    dt = T / N
    u = np.exp(sigma*np.sqrt(dt));  d = 1.0/u
    p = (np.exp((r - q)*dt) - d) / (u - d)
    disc = np.exp(-r*dt)

    # Terminal payoff
    j = np.arange(N + 1)
    ST = S * (u**(N - j)) * (d**j)
    V = np.maximum(K - ST, 0) if kind == "put" else np.maximum(ST - K, 0)

    # Roll back with early-exercise check
    for i in range(N - 1, -1, -1):
        ST = S * (u**(i - np.arange(i + 1))) * (d**np.arange(i + 1))
        V = disc * (p * V[:-1] + (1 - p) * V[1:])
        intrinsic = np.maximum(K - ST, 0) if kind == "put" else np.maximum(ST - K, 0)
        V = np.maximum(V, intrinsic)
    return float(V[0])

am_put_crr = crr_american(TRADE["S"], TRADE["K"], TRADE["r"], TRADE["sigma"],
                          TRADE["T"], TRADE["q"], N=500, kind="put")
print(f"Manual CRR American put (N=500) = ${am_put_crr:.4f}")


In [ ]:
# --- QUANTLIB equivalent — this is exactly what price_american_ql does ---
def ql_american(S, K, r, sigma, T, q=0.0, N=201, kind="put"):
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today
    maturity = today + max(int(T*365 + 0.5), 1)

    spot = ql.QuoteHandle(ql.SimpleQuote(S))
    r_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, r, ql.Actual365Fixed()))
    q_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, q, ql.Actual365Fixed()))
    v_ts = ql.BlackVolTermStructureHandle(
        ql.BlackConstantVol(today, ql.UnitedStates(ql.UnitedStates.NYSE),
                            sigma, ql.Actual365Fixed()))
    process = ql.GeneralizedBlackScholesProcess(spot, q_ts, r_ts, v_ts)

    payoff = ql.PlainVanillaPayoff(ql.Option.Call if kind == "call" else ql.Option.Put, K)
    option = ql.VanillaOption(payoff, ql.AmericanExercise(today, maturity))
    # LR requires odd N. price_american_ql rounds up internally.
    if N % 2 == 0:
        N += 1
    option.setPricingEngine(ql.BinomialVanillaEngine(process, "lr", max(N, 51)))
    return option.NPV()

am_put_ql = ql_american(TRADE["S"], TRADE["K"], TRADE["r"], TRADE["sigma"],
                        TRADE["T"], TRADE["q"], N=201, kind="put")
print(f"QuantLib LR American put (N=201) = ${am_put_ql:.4f}")
print(f"Δ vs CRR manual                  = {am_put_ql - am_put_crr:+.4f}")
print("(LR converges faster — even at lower N it matches a much-finer CRR.)\n")

# Early-exercise premium = American − European
ep_premium = am_put_ql - p_ql_put
print(f"European put (§2)        = ${p_ql_put:.4f}")
print(f"American put (LR tree)   = ${am_put_ql:.4f}")
print(f"Early-exercise premium   = ${ep_premium:.4f}  ({100*ep_premium/p_ql_put:.2f}%)")
print("\n→ pricing an American put with the European formula under-prices by this much.")


---
# §4 — Monte Carlo + Longstaff–Schwartz (American)

### When to use
- **Path-dependent** payoffs (Asian, lookback, autocallable, TARN)
- **Multi-asset** baskets where the curse of dimensionality kills the tree
- You want a **second opinion** on the binomial tree (different numerical regime → catches engine bugs)
- You want path arrays for **risk visualisation** (terminal distribution, max drawdown, etc.)

### Why MC alone can't price American — and what LSM does
A vanilla MC simulator only knows about *terminal* payoffs. American exercise
requires a *backward* decision: at each step, is exercise > continuation?
**Longstaff–Schwartz (2001)** estimates the continuation value with a
cross-sectional regression of discounted future cashflows on a polynomial
basis of the spot — only **in-the-money** paths enter the regression
(out-of-the-money paths are trivially "continue"). The result is an exercise
policy that is *suboptimal but unbiased low* (you can only do worse than
the true policy) → MC LSM gives a **lower bound** on the American price.

### Repo path
`monte_carlo_lsm.price_american` wraps `ql.MCAmericanEngine` with the same
NYSE / Act365 conventions used everywhere else in the project.


In [ ]:
# --- MANUAL: GBM path simulator + Longstaff-Schwartz LSM ---
def gbm_paths(S0, r, sigma, T, q, n_paths, n_steps, antithetic=True):
    dt = T / n_steps
    half = n_paths // 2 if antithetic else n_paths
    Z = np.random.standard_normal((half, n_steps))
    if antithetic:
        Z = np.vstack([Z, -Z])
    drift = (r - q - 0.5*sigma**2) * dt
    diff  = sigma * np.sqrt(dt)
    log_paths = np.cumsum(drift + diff*Z, axis=1)
    paths = S0 * np.exp(np.hstack([np.zeros((Z.shape[0], 1)), log_paths]))
    return paths  # shape (n_paths, n_steps+1)


def lsm_american_put(S0, K, r, sigma, T, q, n_paths=20000, n_steps=50, deg=3):
    """Longstaff-Schwartz American put.

    Invariant
    ---------
    At the START of each iteration ``t``, ``cashflow[i]`` represents path i's
    value at time-step ``t+1`` (or ``n_steps`` initially). The first action
    inside the loop is ``cashflow *= disc_step`` so by the time we use it
    for regression, ``cashflow`` represents value at the CURRENT step ``t``.
    Exercised paths overwrite ``cashflow[idx]`` with intrinsic at step ``t``
    — they are NOT discounted again inside this iteration. After the loop
    ends at ``t=1``, one final ``* disc_step`` brings everything to ``t=0``.

    Why this matters
    ----------------
    The earlier version had two compounding bugs: (a) exercised paths were
    discounted twice — once by the broadcast ``cashflow *= disc_step`` and
    once by a "redundant" overwrite — leaving them one step too far back
    each iteration; (b) the loop terminated at ``t=1`` without a final
    discount, so the returned mean was at ``t=1``, not ``t=0``. Together
    these bias the price by a few cents on a 90d ATM put and corrupt the
    exercise policy on longer-dated trades.
    """
    paths = gbm_paths(S0, r, sigma, T, q, n_paths, n_steps)
    dt = T / n_steps
    disc_step = np.exp(-r*dt)

    # Initial cashflow at step n_steps (terminal payoff).
    cashflow = np.maximum(K - paths[:, -1], 0)

    for t in range(n_steps - 1, 0, -1):
        # Discount everything one step → cashflow is now at step t.
        cashflow = cashflow * disc_step

        S_t = paths[:, t]
        itm = S_t < K
        if itm.sum() < deg + 1:
            continue   # not enough ITM paths to fit the polynomial

        X = S_t[itm]
        Y = cashflow[itm]                     # continuation value, already at step t
        coef = np.polyfit(X, Y, deg)
        cont = np.polyval(coef, X)            # E[continuation | S_t]
        intrinsic = K - X
        exercise_now = intrinsic > cont
        idx = np.where(itm)[0][exercise_now]
        # Replace continuation with exercise payoff — both at step t.
        cashflow[idx] = intrinsic[exercise_now]

    # Loop ended after the t=1 iteration (cashflow at step 1). One more
    # discount brings everything to step 0 = present value.
    cashflow = cashflow * disc_step

    price = cashflow.mean()
    se = cashflow.std(ddof=1) / np.sqrt(n_paths)
    return float(price), float(se)


px_lsm, se_lsm = lsm_american_put(TRADE["S"], TRADE["K"], TRADE["r"],
                                  TRADE["sigma"], TRADE["T"], TRADE["q"],
                                  n_paths=20000, n_steps=50)
print(f"Manual LSM American put = ${px_lsm:.4f}  ± ${1.96*se_lsm:.4f}  (95% CI half-width)")
print(f"Binomial LR (§3)        = ${am_put_ql:.4f}")
print(f"Δ                       = {px_lsm - am_put_ql:+.4f}")
print()
print("Theory: LSM gives a LOWER bound on the true price (suboptimal exercise")
print("policy → unbiased-low in the limit). With finite paths and a polynomial")
print("basis, regression noise can push the sample price either way; what")
print("matters is that the gap sits inside the 95% CI half-width above.")


In [ ]:
# --- QUANTLIB equivalent: ql.MCAmericanEngine ---
def ql_american_mc(S, K, r, sigma, T, q=0.0, kind="put",
                   n_paths=20000, n_steps=50, seed=42):
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today
    maturity = today + max(int(T*365 + 0.5), 1)

    spot = ql.QuoteHandle(ql.SimpleQuote(S))
    r_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, r, ql.Actual365Fixed()))
    q_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, q, ql.Actual365Fixed()))
    v_ts = ql.BlackVolTermStructureHandle(
        ql.BlackConstantVol(today, ql.UnitedStates(ql.UnitedStates.NYSE),
                            sigma, ql.Actual365Fixed()))
    process = ql.GeneralizedBlackScholesProcess(spot, q_ts, r_ts, v_ts)

    payoff = ql.PlainVanillaPayoff(ql.Option.Call if kind == "call" else ql.Option.Put, K)
    option = ql.VanillaOption(payoff, ql.AmericanExercise(today, maturity))

    engine = ql.MCAmericanEngine(
        process, "PseudoRandom",
        timeSteps=n_steps, requiredSamples=n_paths, seed=seed,
    )
    option.setPricingEngine(engine)
    return option.NPV(), option.errorEstimate()

px_qlmc, se_qlmc = ql_american_mc(TRADE["S"], TRADE["K"], TRADE["r"],
                                  TRADE["sigma"], TRADE["T"], TRADE["q"])
print(f"QuantLib MC LSM put     = ${px_qlmc:.4f}  ± ${1.96*se_qlmc:.4f}")
print(f"QuantLib LR tree (§3)   = ${am_put_ql:.4f}    [reference]")
print(f"Δ MC vs tree            = {px_qlmc - am_put_ql:+.4f}")
print("\nThree American prices for the same trade:")
print(f"  Manual LSM:        ${px_lsm:>8.4f}")
print(f"  QuantLib MC LSM:   ${px_qlmc:>8.4f}")
print(f"  QuantLib LR tree:  ${am_put_ql:>8.4f}  (most accurate)")


---
# §5 — Knock-out / knock-in barriers

### When to use which engine
- **Continuous monitoring + flat vol** → `AnalyticBarrierEngine` (Reiner–Rubinstein)
- **Discrete monitoring** (daily / weekly fix) → analytic engine **after BGK shift** on the barrier
- **Smile / skew matters** (any SPX KO) → `FdBlackScholesBarrierEngine` with Dupire local vol

### The BGK shift you must not forget
For discretely-monitored barriers, plug the original B into a continuous formula
and you over-state knock-out probability → under-price the KO product (30–80 bp
on a 1Y / 25-vol DOI call near the barrier — trade-blocking).
$$
B_{\text{adj}} = B \cdot \exp\bigl(\eta \cdot 0.5826 \cdot \sigma \sqrt{\Delta t}\bigr),\quad
\eta = +1 \text{ if } B > S,\; -1 \text{ if } B < S
$$
The constant 0.5826 = $-\zeta(\tfrac12)/\sqrt{2\pi}$ (Broadie–Glasserman–Kou, 1997).

### Parity (free sanity check)
**$\text{KO} + \text{KI} \equiv \text{vanilla}$** for the same K, B, T, σ, r, q.
If your KO + KI doesn't match the vanilla price for the same parameters,
one of your engines is wrong.


In [ ]:
# --- BGK shift + Reiner-Rubinstein knockout (matches knockout.py) ---
BGK = 0.5826  # = -ζ(1/2) / sqrt(2π)

def bgk_shift(B, S, sigma, monitoring_dt):
    if monitoring_dt <= 0 or B == S:
        return B
    eta = 1.0 if B > S else -1.0
    return B * np.exp(eta * BGK * sigma * np.sqrt(monitoring_dt))


def ql_barrier(S, K, B, r, sigma, T, q=0.0, kind="call",
               barrier_kind="out", monitoring="continuous",
               use_local_vol_pde=False, vol_handle=None):
    """KO / KI via QuantLib. Mirrors quantlib_engine.price_knockout_ql.

    ``vol_handle`` (a ``ql.BlackVolTermStructureHandle``) overrides the
    flat ``sigma`` when supplied — the FD engine then uses Dupire local
    vol derived from this surface. With a flat surface, FD ≈ analytic
    (within grid noise). With a real skewed surface, FD ≠ analytic and
    the gap is the *smile correction*.
    """
    monitoring_dt = {"continuous": 0.0, "daily": 1/252, "weekly": 1/52,
                     "monthly": 1/12}.get(monitoring, monitoring) if isinstance(monitoring, str) else monitoring
    # BGK uses the ATM σ — picking a single σ from a surface is desk policy;
    # using the input scalar is the safe pragmatic choice for any cheat sheet.
    B_eff = bgk_shift(B, S, sigma, monitoring_dt)

    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today
    maturity = today + max(int(T*365 + 0.5), 1)

    spot = ql.QuoteHandle(ql.SimpleQuote(S))
    r_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, r, ql.Actual365Fixed()))
    q_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, q, ql.Actual365Fixed()))
    if vol_handle is None:
        v_ts = ql.BlackVolTermStructureHandle(
            ql.BlackConstantVol(today, ql.UnitedStates(ql.UnitedStates.NYSE),
                                sigma, ql.Actual365Fixed()))
    else:
        v_ts = vol_handle
    process = ql.GeneralizedBlackScholesProcess(spot, q_ts, r_ts, v_ts)

    payoff = ql.PlainVanillaPayoff(ql.Option.Call if kind == "call" else ql.Option.Put, K)
    barrier_map = {("out", True): ql.Barrier.DownOut, ("out", False): ql.Barrier.UpOut,
                   ("in",  True): ql.Barrier.DownIn,  ("in",  False): ql.Barrier.UpIn}
    btype = barrier_map[(barrier_kind, B_eff < S)]
    option = ql.BarrierOption(btype, B_eff, 0.0, payoff, ql.EuropeanExercise(maturity))

    if use_local_vol_pde:
        engine = ql.FdBlackScholesBarrierEngine(
            process, 200, 200, 0, ql.FdmSchemeDesc.Douglas(),
            True,    # localVol
            0.01,    # σ² floor on illiquid wings
        )
    else:
        engine = ql.AnalyticBarrierEngine(process)
    option.setPricingEngine(engine)
    return float(option.NPV())


# Worked example: 90-day SPX up-and-out call, barrier 5% above spot, daily-monitored.
B_up = round(TRADE["S"] * 1.05, 2)
ko_call = ql_barrier(TRADE["S"], TRADE["K"], B_up, TRADE["r"], TRADE["sigma"],
                     TRADE["T"], TRADE["q"], kind="call",
                     barrier_kind="out", monitoring="daily")
ki_call = ql_barrier(TRADE["S"], TRADE["K"], B_up, TRADE["r"], TRADE["sigma"],
                     TRADE["T"], TRADE["q"], kind="call",
                     barrier_kind="in", monitoring="daily")
vanilla_call = p_ql_call

print(f"Spot S = {TRADE['S']:,.2f}   Strike K = {TRADE['K']:,.2f}   Barrier B = {B_up:,.2f} (UP, +5%)")
print(f"Monitoring: daily (BGK-adjusted from B={B_up:.2f} to {bgk_shift(B_up, TRADE['S'], TRADE['sigma'], 1/252):.4f})\n")
print(f"  Up-and-Out call (KO)  = ${ko_call:.4f}")
print(f"  Up-and-In  call (KI)  = ${ki_call:.4f}")
print(f"  Vanilla call (§2)     = ${vanilla_call:.4f}")
print(f"  KO + KI               = ${ko_call + ki_call:.4f}")
print(f"  parity diff           = {ko_call + ki_call - vanilla_call:+.2e}")
print("\n→ KO and KI sum to vanilla (within numerical noise). Parity validated.")


In [ ]:
# --- Local-vol PDE — the SMILE CORRECTION made visible ---
# Build a synthetic SPX-style skew anchored on the trade's ATM σ. The skew
# shape (puts richer than ATM, calls cheaper) reflects the typical equity
# index surface; magnitude is desk-pickable. Replace this matrix with
# bid/ask vols pulled from yf.Ticker("SPY").options (or your IV surface
# service) for a market-marked surface — the rest of the code is identical.
sigma_atm = TRADE["sigma"]
strikes = np.array([0.90, 1.00, 1.10]) * TRADE["S"]
tenors_days = [30, 90, 180]

# Rows = strikes (ascending), cols = tenors (ascending). Negative skew:
# downside +4-5 vol points, upside −2-3 vol points. Long-dated flatter.
vol_matrix_np = np.array([
    # 30d                  90d                  180d
    [sigma_atm + 0.045, sigma_atm + 0.035, sigma_atm + 0.025],   # 90 % strike (deep OTM put)
    [sigma_atm,         sigma_atm,         sigma_atm + 0.005],   # 100 % strike (ATM)
    [sigma_atm - 0.030, sigma_atm - 0.025, sigma_atm - 0.015],   # 110 % strike (OTM call)
])
print(f"ATM anchor σ = {sigma_atm:.2%}    skew shape (vol points vs ATM):")
for i, s in enumerate([0.90, 1.00, 1.10]):
    print(f"  {s*100:.0f}% strike: " + "  ".join(
        f"{(vol_matrix_np[i,j] - sigma_atm)*100:+.1f}vp" for j in range(3)
    ))

today = ql.Date.todaysDate()
ql.Settings.instance().evaluationDate = today
expiries = [today + d for d in tenors_days]

ql_vols = ql.Matrix(len(strikes), len(expiries))
for i in range(len(strikes)):
    for j in range(len(expiries)):
        ql_vols[i][j] = float(vol_matrix_np[i, j])

# BlackVarianceSurface interpolates Black variances bilinearly on (T, K).
# This is what every desk uses as the input to a Dupire local-vol build.
surface = ql.BlackVarianceSurface(
    today, ql.UnitedStates(ql.UnitedStates.NYSE),
    expiries, [float(k) for k in strikes],
    ql_vols, ql.Actual365Fixed(),
)
surface.enableExtrapolation()
smile_handle = ql.BlackVolTermStructureHandle(surface)

# Same up-and-out call. Both FD prices use the SAME ATM σ as the analytic
# engine above — so the gap is purely the skew effect, not an apples-vs-
# oranges comparison of different vol levels.
ko_flat_fd = ql_barrier(TRADE["S"], TRADE["K"], B_up, TRADE["r"], sigma_atm,
                        TRADE["T"], TRADE["q"], kind="call",
                        barrier_kind="out", monitoring="daily",
                        use_local_vol_pde=True, vol_handle=None)

ko_smile_fd = ql_barrier(TRADE["S"], TRADE["K"], B_up, TRADE["r"], sigma_atm,
                         TRADE["T"], TRADE["q"], kind="call",
                         barrier_kind="out", monitoring="daily",
                         use_local_vol_pde=True, vol_handle=smile_handle)

print("\nSame KO call, three vol regimes:")
print(f"  Analytic (flat-σ ATM)            = ${ko_call:.4f}    [from previous cell]")
print(f"  FD       (flat-σ ATM)            = ${ko_flat_fd:.4f}    ← validates analytic ≈ FD on flat surface")
print(f"  FD       (skewed BlackVarianceSurface) = ${ko_smile_fd:.4f}")
print(f"  SMILE CORRECTION                  = ${ko_smile_fd - ko_flat_fd:+.4f}  ({1e4*(ko_smile_fd - ko_flat_fd)/max(ko_flat_fd,1e-6):+.0f} bp)")
print()
print("Direction is intuitive: this is an UP-and-OUT call. With negative skew,")
print("upside vol is LOWER than ATM → fewer paths touch the upper barrier →")
print("knockout probability falls → the KO call is worth MORE under skew.")
print("The analytic engine collapses any surface to one σ and misses this entirely.")

# Keep the legacy variable name for the §9 summary table.
ko_local = ko_smile_fd


---
# §6 — Discrete cash dividends (American option, FDM)

### When to use
- Single-name equity with **scheduled cash dividends** (most US blue-chips)
- American exercise — especially **calls**: the early-exercise decision flips
  *just before* an ex-div date when the dividend > remaining time-value
- A continuous-q approximation **cannot** reproduce the discrete spot drop
  on the ex-div date. Mixing continuous-q + discrete cash on the same name
  is a model error (double-counts the dividend)

### Engine
`ql.FdBlackScholesVanillaEngine(process, dividendSchedule, n_t, n_x)` with
`q ≡ 0` in the process and the cash dividends in a `DividendSchedule`.
Repo wrapper: `quantlib_engine.price_american_discrete_div_ql`.


In [ ]:
def ql_american_discrete_div(S, K, r, sigma, T, dividends, kind="call",
                              n_t=200, n_x=200):
    """dividends: list of (ql.Date, float). Continuous q is set to 0."""
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today
    maturity = today + max(int(T*365 + 0.5), 1)

    spot = ql.QuoteHandle(ql.SimpleQuote(S))
    r_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, r, ql.Actual365Fixed()))
    q_ts = ql.YieldTermStructureHandle(ql.FlatForward(today, 0.0, ql.Actual365Fixed()))
    v_ts = ql.BlackVolTermStructureHandle(
        ql.BlackConstantVol(today, ql.UnitedStates(ql.UnitedStates.NYSE),
                            sigma, ql.Actual365Fixed()))
    process = ql.GeneralizedBlackScholesProcess(spot, q_ts, r_ts, v_ts)

    payoff = ql.PlainVanillaPayoff(ql.Option.Call if kind == "call" else ql.Option.Put, K)
    option = ql.VanillaOption(payoff, ql.AmericanExercise(today, maturity))

    sched = ql.DividendSchedule()
    for d, amt in dividends:
        sched.append(ql.FixedDividend(float(amt), d))

    option.setPricingEngine(ql.FdBlackScholesVanillaEngine(process, sched, n_t, n_x))
    return float(option.NPV())


# SPY pays a quarterly dividend; SPX is just a yield. We illustrate the API
# on a synthetic single dividend mid-life. Replace with the real ex-div schedule
# when pricing a single-name option (e.g. AAPL, MSFT).
today = ql.Date.todaysDate()
mid_life = today + max(int(TRADE["T"]*365 / 2), 1)
synthetic_div = [(mid_life, 1.50)]  # $1.50 cash dividend at T/2

px_disc = ql_american_discrete_div(
    TRADE["S"], TRADE["K"], TRADE["r"], TRADE["sigma"], TRADE["T"],
    synthetic_div, kind="call",
)
px_cont = ql_american(TRADE["S"], TRADE["K"], TRADE["r"], TRADE["sigma"],
                      TRADE["T"], TRADE["q"], kind="call")  # uses continuous q

print(f"American CALL — same trade, two dividend models")
print(f"  continuous-q   = ${px_cont:.4f}")
print(f"  discrete cash  = ${px_disc:.4f}  (single $1.50 div at T/2)")
print(f"  difference     = {px_disc - px_cont:+.4f}")
print("\n→ For single-name options, use discrete-cash; for SPX/index, continuous-q is fine.")


---
# §7 — Greeks: analytic vs bump-reprice vs FDM

### Three flavours, three trade-offs

| Method | Cost | Accuracy | When to prefer |
|---|---|---|---|
| **Analytic** | one repricing | exact (closed form) | European vanillas only |
| **Bump-reprice** | 2× per Greek | engine-dependent noise | Anything not analytic; default for barrier engines |
| **FDM (engine-native)** | one solve, free Greeks | smooth, grid-controlled | American risk reporting, hedging |

### Why the repo uses FDM (not LR-tree bump) for American Greeks
LR-tree bump-reprice produces **"ghost gamma"** — adjacent strikes can show
30 %+ different bump-γ because LR snaps node positions to discrete grid points.
FDM uses a fixed space/time mesh and interpolates → smooth Greek surfaces
suitable for risk reporting and delta-hedging. See
`quantlib_engine.greeks_american_fdm_ql`.

### Desk practice: two Greeks for two purposes
- **Risk reporting / pre-trade hedging:** FDM Greeks. Smooth across strikes,
  no node-snapping artefacts, suitable for Greeks ladders and hedge ratios.
- **P&L Explain (T+1 attribution):** **bump-reprice with the exact engine
  used for the mark.** If the official mark came off `BinomialVanillaEngine`,
  bump that same engine — the Greeks then reconcile cleanly to the price
  delta the trader sees. Mixing FDM Greeks against a tree-marked book leaves
  unexplained residual P&L.

### Why barrier vega ≠ "shift the smile"
For a barrier option under a real surface, "vega" has two definitions:
(1) bump σ on a flat-vol process and reprice (parallel shift), (2) shift the
whole surface and reprice. Desks generally report (1) and quote (2) separately
as **smile vega**. The repo's `greeks_knockout_ql` does (1) for stability —
FD discretisation noise on small bumps under local-vol pricing routinely
flips the sign of γ.

### Theta sign convention (sanity check)
The repo defines θ = `(V(T-Δt) - V(T))` per calendar day. For a long-γ
position this is **negative** ("decay") — what a trader actually sees on
the daily P&L. Be careful with QL: `option.theta()` returns ∂V/∂t in
*years*; we divide by 365 to get the per-day desk convention.


In [ ]:
# Greek-method comparison on the same European put.
S, K, r, sigma, T, q = TRADE["S"], TRADE["K"], TRADE["r"], TRADE["sigma"], TRADE["T"], TRADE["q"]

# --- 1) Analytic (closed form) ---
d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
d2 = d1 - sigma*np.sqrt(T)
greeks_analytic = {
    "delta": -np.exp(-q*T) * norm.cdf(-d1),
    "gamma": np.exp(-q*T) * norm.pdf(d1) / (S*sigma*np.sqrt(T)),
    "vega":  S*np.exp(-q*T) * norm.pdf(d1) * np.sqrt(T) / 100.0,
    "theta": (-S*np.exp(-q*T)*norm.pdf(d1)*sigma/(2*np.sqrt(T))
              + r*K*np.exp(-r*T)*norm.cdf(-d2)
              - q*S*np.exp(-q*T)*norm.cdf(-d1)) / 365.0,
    "rho":  -K*T*np.exp(-r*T) * norm.cdf(-d2) / 100.0,
}

# --- 2) Bump-reprice (central difference) on the analytic engine ---
def reprice(S_, K_, r_, sigma_, T_, q_):
    return ql_european(S_, K_, r_, sigma_, T_, q_, "put")[0]

h_S = 0.005 * S
greeks_bump = {
    "delta": (reprice(S+h_S, K, r, sigma, T, q) - reprice(S-h_S, K, r, sigma, T, q)) / (2*h_S),
    "gamma": (reprice(S+h_S, K, r, sigma, T, q) - 2*reprice(S, K, r, sigma, T, q)
              + reprice(S-h_S, K, r, sigma, T, q)) / (h_S**2),
    "vega":  (reprice(S, K, r, sigma+0.005, T, q) - reprice(S, K, r, sigma-0.005, T, q)) / (2*0.005) * 0.01,
    "theta": (reprice(S, K, r, sigma, max(T-1/365, 1e-6), q) - reprice(S, K, r, sigma, T, q)),
    "rho":   (reprice(S, K, r+1e-4, sigma, T, q) - reprice(S, K, r-1e-4, sigma, T, q)) / (2e-4) * 0.01,
}

# --- 3) Engine-native (QL AnalyticEuropeanEngine exposes all of them) ---
_, greeks_engine = ql_european(S, K, r, sigma, T, q, "put")

# Compare
print(f"{'Greek':6s} {'Analytic':>12s} {'Bump':>12s} {'Engine':>12s} {'max |diff|':>12s}")
print("-"*60)
for k in ("delta", "gamma", "vega", "theta", "rho"):
    a, b, e = greeks_analytic[k], greeks_bump[k], greeks_engine[k]
    print(f"{k:6s} {a:>12.6f} {b:>12.6f} {e:>12.6f} {max(abs(a-b),abs(a-e)):>12.2e}")
print("\n→ All three agree to ≤1e-3 — the European reference is rock-solid.")
print("  For American/barrier, only bump-reprice + FDM are available; LR-bump γ is unreliable.")


---
# §8 — Engine router (decision tree)

The repo's `src/engines/router.py` maps an `option_type` string to the right
pricer. The decision tree below is what it implements; use the same logic in
your own scripts when you don't want to import the package.

```text
                ┌──────────────────────────┐
                │   What is the payoff?    │
                └────────────┬─────────────┘
        ┌────────────────────┼─────────────────────┐
        │                    │                     │
   European vanilla     American vanilla       Barrier (KO/KI)
        │                    │                     │
   AnalyticEuropean     ┌────┴─────┐         ┌─────┴─────────┐
   Engine               │          │         │               │
                  cash divs?   yield only   discrete         smile?
                       │          │          monit?              │
                       ▼          ▼            │                 ▼
                FdBlackScholes  Binomial      BGK shift   FdBlackScholes
                Vanilla         "lr"          → Analytic  Barrier (localVol)
                (DivSched)      tree          Barrier
```


In [ ]:
def pick_engine(option_type, has_smile=False, monitoring="continuous",
                cash_dividends=False):
    """Plain-Python translation of src/engines/router.py decision logic."""
    ot = option_type.lower()
    if ot.startswith("european"):
        return "AnalyticEuropeanEngine"
    if ot.startswith("american"):
        if cash_dividends:
            return "FdBlackScholesVanillaEngine + DividendSchedule"
        return 'BinomialVanillaEngine(process, "lr", N)'
    if ot.startswith(("knockout", "knockin")):
        if has_smile:
            return "FdBlackScholesBarrierEngine (localVol=True)"
        if monitoring != "continuous":
            return "AnalyticBarrierEngine + BGK barrier shift"
        return "AnalyticBarrierEngine"
    raise ValueError(f"unknown option_type {option_type!r}")


for cfg in [
    ("european_put",  False, "continuous", False),
    ("american_put",  False, "continuous", False),
    ("american_call", False, "continuous", True),
    ("knockout_call", False, "daily",      False),
    ("knockout_call", True,  "continuous", False),
    ("knockin_put",   False, "continuous", False),
]:
    print(f"  {cfg[0]:14s}  smile={cfg[1]!s:5s} mon={cfg[2]:10s} cashdiv={cfg[3]!s:5s} -> {pick_engine(*cfg)}")


---
# §9 — One-shot summary table

Every method ran against the same live `TRADE`. Use this as the cheat-sheet
landing pad — re-run the notebook top-to-bottom and the numbers below
refresh from live yfinance data automatically.


In [ ]:
rows = [
    ("§2 Black-Scholes (manual)",     "European put",    p_manual_put, f"σ={TRADE['sigma']:.2%}",        "scipy.stats.norm"),
    ("§2 AnalyticEuropeanEngine",     "European put",    p_ql_put,     f"σ={TRADE['sigma']:.2%}",        "QuantLib analytic"),
    ("§3 CRR tree (manual, N=500)",   "American put",    am_put_crr,   f"σ={TRADE['sigma']:.2%}",        "manual CRR"),
    ("§3 BinomialVanillaEngine 'lr'", "American put",    am_put_ql,    f"σ={TRADE['sigma']:.2%}",        "QuantLib LR tree"),
    ("§4 Manual LSM",                 "American put",    px_lsm,       f"σ={TRADE['sigma']:.2%}, 20k pths", "manual MC"),
    ("§4 MCAmericanEngine",           "American put",    px_qlmc,      f"σ={TRADE['sigma']:.2%}, 20k pths", "QuantLib MC LSM"),
    ("§5 KO (analytic, daily+BGK)",   "Up-and-Out call", ko_call,      f"σ={TRADE['sigma']:.2%} flat",   "AnalyticBarrierEngine"),
    ("§5 KI (analytic, daily+BGK)",   "Up-and-In  call", ki_call,      f"σ={TRADE['sigma']:.2%} flat",   "AnalyticBarrierEngine"),
    ("§5 KO (FD + skewed surface)",   "Up-and-Out call", ko_local,     f"ATM={TRADE['sigma']:.2%} + skew", "FdBlackScholesBarrierEngine"),
    ("§6 American + discrete div",    "American call",   px_disc,      f"σ={TRADE['sigma']:.2%}, $1.50@T/2", "FdBlackScholesVanillaEngine"),
]

print(f"\nTRADE: S={TRADE['S']:,.2f}  K={TRADE['K']:,.2f}  T={TRADE['T']*365:.0f}d  σ={TRADE['sigma']:.2%}  r={TRADE['r']:.2%}  q={TRADE['q']:.2%}")
print(f"Rate tenor: {TRADE.get('rate_tenor','?')}    Date: {TRADE['ref_date']}    Source: {TRADE['source']}\n")
print(f"{'Method':36s} {'Payoff':18s} {'Price':>10s}  {'Vol regime':22s}  Engine")
print("-"*120)
for label, payoff, px, vol_note, eng in rows:
    print(f"{label:36s} {payoff:18s} ${px:>9.4f}  {vol_note:22s}  {eng}")
print("-"*120)
print("\nCross-checks performed in the notebook:")
print(f"  • Put-call parity: C - P = S e^-qT - K e^-rT  →  diff = {(p_ql_call - p_ql_put) - (TRADE['S']*np.exp(-TRADE['q']*TRADE['T']) - TRADE['K']*np.exp(-TRADE['r']*TRADE['T'])):.2e}")
print(f"  • Manual BS vs QL AnalyticEuropeanEngine     →  diff = {p_manual_put - p_ql_put:.2e}")
print(f"  • KO + KI = vanilla call (analytic, same σ)  →  diff = {(ko_call + ki_call) - p_ql_call:.2e}")
print(f"  • Smile correction (FD-flat → FD-skew)       →  Δ = ${ko_smile_fd - ko_flat_fd:+.4f} ({1e4*(ko_smile_fd - ko_flat_fd)/max(ko_flat_fd,1e-6):+.0f} bp)")
print("\nThis notebook is the single source of truth for engine selection in this repo.")
